# TBD Phase 2: Performance & Computing Models

## Introduction
In this lab, you will compare the performance and computing models of four popular data processing engines: **Polars, Pandas, DuckDB, and PySpark**.

You will explore:
- **Performance**: Single-node processing speed, parallel execution, and memory usage.
- **Scalability**: How performance changes with the number of cores (single-node) and executors (cluster).
- **Computing Models**: Out-of-core vs. In-memory processing, and Eager vs. Lazy execution.

### Engine Capabilities
The following table summarizes the key capabilities of the engines we will be testing. Use this as a reference.

| Engine | Query Optimizer | Distributed | Arrow-backed | Out-of-Core | Parallel | APIs | GPU Support |
|---|---|---|---|---|--|---|---|
| **Pandas** | ❌ | ❌ | optional ≥ 2.0 | ❌ | ❌ | DataFrame | ❌ |
| **Polars** | ✅ | ❌ | ✅ | ✅ | ✅ | DataFrame | ✅ (opt) |
| **PySpark** | ✅ | ✅ | Pandas UDF/IO | ✅ | ✅ | SQL, DataFrame | ❌ (no GPU) |
| **DuckDB** | ✅ | ❌ | ✅ | ✅ | ❌ | SQL, Relational API | ❌ |

## Prerequisites
Ensure you have the necessary libraries installed.

In [1]:
%pip install polars pandas duckdb pyspark faker deltalake memory_profiler pyarrow

  Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.4/455.4 MB 52.8 MB/s  0:00:07:00:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached wrapt-2.0.1-cp310-cp310-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (9.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.7/805.7 kB 18.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 MB 61.2 MB/s  0:00:00 eta 0:00:01
Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/

In [2]:
import polars as pl
import pandas as pd
import duckdb
from pyspark.sql import SparkSession
from faker import Faker
import numpy as np
import os
import time
import psutil
from memory_profiler import memory_usage

# Initialize Spark (Single Node)
spark = SparkSession.builder \
    .appName("BigDataLab2") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

## Part 1: Data Generation

We will generate a synthetic dataset simulating social media posts with a rich schema.

**Schema**:
- `post_id` (String): Unique identifier.
- `user_id` (Integer): User identifier.
- `timestamp` (DateTime): Time of post.
- `content` (String): Text content.
- `likes` (Integer): Number of likes.
- `views` (Integer): Number of views.
- `category` (String): Post category.
- `tags` (List[String]): Hashtags.
- `location` (String): User location.
- `device` (String): Device used (Mobile, Web, etc.).
- `latency` (Float): Network latency.
- `error_rate` (Float): Error rate during upload.

In [3]:
def generate_data(num_records=1_000_000, output_path="social_media_data.parquet"):
    fake = Faker()
    
    print(f"Generating {num_records} records...")
    
    # Generate data using numpy for speed where possible
    data = {
        "post_id": [fake.uuid4() for _ in range(num_records)],
        "user_id": np.random.randint(1, 100_000, num_records),
        "timestamp": pd.date_range(start="2023-01-01", periods=num_records, freq="s").to_numpy().astype("datetime64[us]"),
        "likes": np.random.randint(0, 10_000, num_records),
        "views": np.random.randint(0, 1_000_000, num_records),
        "category": np.random.choice(["Tech", "Health", "Travel", "Food", "Fashion", "Politics", "Sports"], num_records),
        "tags": [np.random.choice(["#viral", "#new", "#trending", "#hot", "#update"], size=np.random.randint(1, 4)).tolist() for _ in range(num_records)],
        "location": np.random.choice(["USA", "UK", "DE", "PL", "FR", "JP", "BR"], num_records),
        "device": np.random.choice(["Mobile", "Desktop", "Tablet"], num_records),
        "latency": np.random.uniform(10.0, 500.0, num_records),
        "error_rate": np.random.beta(1, 10, num_records),
        "content": [fake.sentence() for _ in range(min(num_records, 1000))] * (num_records // 1000 + 1)
    }
    
    # Trim to exact size
    data["content"] = data["content"][:num_records]
    
    df = pd.DataFrame(data)
    
    print("Writing to Parquet...")
    df.to_parquet(output_path, engine="pyarrow")
    print(f"Data saved to {output_path}")

# Generate 5 million records
generate_data(num_records=5_000_000)

Generating 5000000 records...
Writing to Parquet...
Data saved to social_media_data.parquet


## Part 2: Measuring Performance

### 2.1 Execution Time
Use `%time` or `%timeit` to measure execution time.

In [4]:
# Example: Measuring time for all engines
print("--- Performance Benchmark Example ---")

# Pandas
print("Pandas Load Time:")
%time df_pd = pd.read_parquet("social_media_data.parquet")

# Polars
print("\nPolars Load Time:")
%time df_pl = pl.read_parquet("social_media_data.parquet")

# DuckDB
print("\nDuckDB Query Time:")
%time duckdb.sql("SELECT count(*) FROM 'social_media_data.parquet'").show()

# PySpark
print("\nSpark Load Time:")
%time df_spark = spark.read.parquet("social_media_data.parquet"); df_spark.count()

--- Performance Benchmark Example ---
Pandas Load Time:
CPU times: user 5.9 s, sys: 3.45 s, total: 9.35 s
Wall time: 5.59 s

Polars Load Time:
CPU times: user 806 ms, sys: 486 ms, total: 1.29 s
Wall time: 290 ms

DuckDB Query Time:
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      5000000 │
└──────────────┘

CPU times: user 8.67 ms, sys: 8.41 ms, total: 17.1 ms
Wall time: 7.65 ms

Spark Load Time:


CPU times: user 10.7 ms, sys: 16.5 ms, total: 27.2 ms
Wall time: 5.52 s


5000000

## Part 3: Student Tasks

### Task 1: Performance & Scalability (Single Node)

**Goal**: Benchmark the engines and test how they scale with available CPU cores.

**Instructions**:
1.  **Define Queries**: Create 3 distinct queries of your own choice. They should cover:
    -   **Query A**: A simple aggregation (e.g., grouping by a categorical column and calculating means).
    -   **Query B**: A window function or more complex transformation.
    -   **Query C**: A join (e.g., self-join or join with a smaller generated table) with filtering.
2.  **Benchmark**: Implement these queries in **Pandas, Polars, DuckDB, and PySpark**.
    -   Measure **Execution Time** using `%time` or `time.time()`.
    -   Measure **Peak Memory** usage using `memory_profiler` (e.g., `memory_usage()`).
3.  **Scalability Test**: 
    -   Select **all engines** that support parallel execution on a single node (e.g., Polars, DuckDB).
    -   Run **all 3 queries** with different numbers of threads/cores (e.g., 1, 2, 4, 8).
    -   Plot the speedup for each query and engine.

**Tip**: 
-   Polars: [polars.thread_pool_size](https://docs.pola.rs/api/python/stable/reference/api/polars.thread_pool_size.html) Please also note that *Thread configuration in Polars requires process restart*
-   DuckDB: `PRAGMA threads=n`
-   Spark: `master="local[n]"`

In [8]:
import os
import gc
import time
import math
import duckdb
import pandas as pd
import polars as pl

from memory_profiler import memory_usage
from pyspark.sql import SparkSession, functions as F, Window

DATA_PATH = "social_media_data.parquet"

# ---------------------------
# Helpers: timing + memory
# ---------------------------
def _gc():
    gc.collect()

def measure(func, repeats=3, warmup=1):
    """
    Returns dict: time_min_s, time_mean_s, peak_mem_mb
    Peak memory measured for Python process (memory_profiler).
    """
    # Warmup
    for _ in range(warmup):
        _gc()
        func()

    times = []
    peaks = []
    for _ in range(repeats):
        _gc()

        t0 = time.time()
        peak = memory_usage((func, ), max_usage=True, interval=0.1, timeout=None)
        t1 = time.time()

        times.append(t1 - t0)
        peaks.append(float(peak))

    return {
        "time_min_s": float(min(times)),
        "time_mean_s": float(sum(times)/len(times)),
        "peak_mem_mb": float(max(peaks)),
    }

def pretty_print_df(df, n=5, name="result"):
    print(f"\n--- {name} ---")
    print(df.head(n))
    print("rows:", len(df))

# ---------------------------
# Define Queries A/B/C
# ---------------------------
# Assumption: generated data has these columns (typical for the provided generator):
# user_id, post_id, timestamp, category, location, likes, views, shares, comment_count
# If your generator differs, adjust column names in one place below.

COL_USER = "user_id"
COL_POST = "post_id"
COL_TS   = "timestamp"
COL_CAT  = "category"
COL_LOC  = "location"
COL_LIKES = "likes"
COL_VIEWS = "views"
COL_SHARES = "shares"
COL_COMMENTS = "comment_count"

# Query A (aggregation): "For each category+location: avg likes, sum views, count posts for popular posts"
# Filter: views >= 100
def query_a_pandas(df: pd.DataFrame) -> pd.DataFrame:
    d = df[df[COL_VIEWS] >= 100]
    out = (d.groupby([COL_CAT, COL_LOC], as_index=False)
             .agg(avg_likes=(COL_LIKES, "mean"),
                  sum_views=(COL_VIEWS, "sum"),
                  posts=("post_id", "count")))
    return out.sort_values(["sum_views"], ascending=False)

def query_b_pandas(df: pd.DataFrame) -> pd.DataFrame:
    # Query B (window/topN): "Top 3 posts per category by likes"
    d = df[[COL_CAT, COL_POST, COL_LIKES]].copy()
    d["rn"] = d.groupby(COL_CAT)[COL_LIKES].rank(method="first", ascending=False)
    out = d[d["rn"] <= 3].sort_values([COL_CAT, "rn"])
    return out

def query_c_pandas(df: pd.DataFrame) -> pd.DataFrame:
    # Query C (join): "Join with small dim table mapping category->weight; compute weighted likes and aggregate"
    dim = pd.DataFrame({
        COL_CAT: df[COL_CAT].dropna().unique()[:50],  # keep it small
    })
    dim["weight"] = (pd.Series(range(1, len(dim)+1)) % 7 + 1).astype(float)

    joined = df.merge(dim, on=COL_CAT, how="inner")
    joined["wlikes"] = joined[COL_LIKES] * joined["weight"]
    out = (joined.groupby(COL_CAT, as_index=False)
                 .agg(weighted_likes=("wlikes", "sum"),
                      posts=(COL_POST, "count")))
    return out.sort_values("weighted_likes", ascending=False)

# Polars equivalents
def query_a_polars_eager(df: pl.DataFrame) -> pl.DataFrame:
    return (df.filter(pl.col(COL_VIEWS) >= 100)
              .group_by([COL_CAT, COL_LOC])
              .agg([
                  pl.col(COL_LIKES).mean().alias("avg_likes"),
                  pl.col(COL_VIEWS).sum().alias("sum_views"),
                  pl.count().alias("posts"),
              ])
              .sort("sum_views", descending=True))

def query_b_polars_eager(df: pl.DataFrame) -> pl.DataFrame:
    # top 3 per category by likes
    return (df.select([COL_CAT, COL_POST, COL_LIKES])
              .with_columns(
                  pl.col(COL_LIKES).rank("dense", descending=True).over(COL_CAT).alias("rk")
              )
              .filter(pl.col("rk") <= 3)
              .sort([COL_CAT, "rk"]))

def query_c_polars_eager(df: pl.DataFrame) -> pl.DataFrame:
    cats = df.select(COL_CAT).unique().head(50)
    dim = cats.with_columns((pl.arange(0, cats.height) % 7 + 1).cast(pl.Float64).alias("weight"))
    joined = df.join(dim, on=COL_CAT, how="inner").with_columns((pl.col(COL_LIKES) * pl.col("weight")).alias("wlikes"))
    return (joined.group_by(COL_CAT)
                  .agg([
                      pl.col("wlikes").sum().alias("weighted_likes"),
                      pl.count().alias("posts"),
                  ])
                  .sort("weighted_likes", descending=True))

def query_a_polars_lazy(scan: pl.LazyFrame) -> pl.DataFrame:
    return (scan.filter(pl.col(COL_VIEWS) >= 100)
                .group_by([COL_CAT, COL_LOC])
                .agg([
                    pl.col(COL_LIKES).mean().alias("avg_likes"),
                    pl.col(COL_VIEWS).sum().alias("sum_views"),
                    pl.count().alias("posts"),
                ])
                .sort("sum_views", descending=True)
                .collect())

def query_b_polars_lazy(scan: pl.LazyFrame) -> pl.DataFrame:
    return (scan.select([COL_CAT, COL_POST, COL_LIKES])
                .with_columns(
                    pl.col(COL_LIKES).rank("dense", descending=True).over(COL_CAT).alias("rk")
                )
                .filter(pl.col("rk") <= 3)
                .sort([COL_CAT, "rk"])
                .collect())

def query_c_polars_lazy(scan: pl.LazyFrame) -> pl.DataFrame:
    # build tiny dim in eager, then join in lazy
    cats = scan.select(COL_CAT).unique().limit(50).collect()
    dim = cats.with_columns((pl.arange(0, cats.height) % 7 + 1).cast(pl.Float64).alias("weight")).lazy()
    return (scan.join(dim, on=COL_CAT, how="inner")
                .with_columns((pl.col(COL_LIKES) * pl.col("weight")).alias("wlikes"))
                .group_by(COL_CAT)
                .agg([
                    pl.col("wlikes").sum().alias("weighted_likes"),
                    pl.count().alias("posts"),
                ])
                .sort("weighted_likes", descending=True)
                .collect())

# DuckDB SQL
DUCK_A = f"""
SELECT {COL_CAT} AS category, {COL_LOC} AS location,
       AVG({COL_LIKES}) AS avg_likes,
       SUM({COL_VIEWS}) AS sum_views,
       COUNT(*) AS posts
FROM read_parquet('{DATA_PATH}')
WHERE {COL_VIEWS} >= 100
GROUP BY 1,2
ORDER BY sum_views DESC
"""

DUCK_B = f"""
SELECT *
FROM (
  SELECT {COL_CAT} AS category, {COL_POST} AS post_id, {COL_LIKES} AS likes,
         DENSE_RANK() OVER (PARTITION BY {COL_CAT} ORDER BY {COL_LIKES} DESC) AS rk
  FROM read_parquet('{DATA_PATH}')
)
WHERE rk <= 3
ORDER BY category, rk
"""

DUCK_C = f"""
WITH dim AS (
  SELECT DISTINCT {COL_CAT} AS category,
         (row_number() OVER () % 7 + 1)::DOUBLE AS weight
  FROM read_parquet('{DATA_PATH}')
  LIMIT 50
)
SELECT d.category,
       SUM(t.{COL_LIKES} * d.weight) AS weighted_likes,
       COUNT(*) AS posts
FROM read_parquet('{DATA_PATH}') t
JOIN dim d ON t.{COL_CAT} = d.category
GROUP BY 1
ORDER BY weighted_likes DESC
"""

# Spark builders (local session per cores)
def build_spark(local_cores: int, driver_mem="4g"):
    return (SparkSession.builder
            .appName(f"TBD-Phase2-local[{local_cores}]")
            .master(f"local[{local_cores}]")
            .config("spark.driver.memory", driver_mem)
            .getOrCreate())

def spark_query_a(df):
    return (df.where(F.col(COL_VIEWS) >= 100)
              .groupBy(COL_CAT, COL_LOC)
              .agg(F.avg(COL_LIKES).alias("avg_likes"),
                   F.sum(COL_VIEWS).alias("sum_views"),
                   F.count(F.lit(1)).alias("posts"))
              .orderBy(F.col("sum_views").desc()))

def spark_query_b(df):
    w = Window.partitionBy(COL_CAT).orderBy(F.col(COL_LIKES).desc())
    return (df.select(COL_CAT, COL_POST, COL_LIKES)
              .withColumn("rk", F.dense_rank().over(w))
              .where(F.col("rk") <= 3)
              .orderBy(COL_CAT, "rk"))

def spark_query_c(df):
    dim = (df.select(COL_CAT).distinct().limit(50)
             .withColumn("weight", (F.row_number().over(Window.orderBy(COL_CAT)) % 7 + 1).cast("double")))
    j = (df.join(dim, on=COL_CAT, how="inner")
           .withColumn("wlikes", F.col(COL_LIKES) * F.col("weight")))
    return (j.groupBy(COL_CAT)
              .agg(F.sum("wlikes").alias("weighted_likes"),
                   F.count(F.lit(1)).alias("posts"))
              .orderBy(F.col("weighted_likes").desc()))

# ---------------------------
# Load inputs once (where appropriate)
# ---------------------------
print("Loading Pandas + Polars eager frames...")
pdf = pd.read_parquet(DATA_PATH)
pldf = pl.read_parquet(DATA_PATH)
plscan = pl.scan_parquet(DATA_PATH)

print("Rows (pandas):", len(pdf), "Rows (polars):", pldf.height)

# ---------------------------
# Benchmark runner
# ---------------------------
results = []

def add_result(engine, query_id, cores, metrics, rows_out):
    results.append({
        "engine": engine,
        "query": query_id,
        "cores": cores,
        **metrics,
        "rows_out": int(rows_out),
    })

# Pandas (cores not applicable)
for qid, qfn in [("A", lambda: query_a_pandas(pdf)),
                ("B", lambda: query_b_pandas(pdf)),
                ("C", lambda: query_c_pandas(pdf))]:
    print(f"\nRunning Pandas query {qid}...")
    out_holder = {}
    def _run():
        out_holder["out"] = qfn()
        return out_holder["out"]
    m = measure(_run, repeats=2, warmup=1)
    add_result("pandas", qid, None, m, len(out_holder["out"]))

# Polars eager
for qid, qfn in [("A", lambda: query_a_polars_eager(pldf)),
                ("B", lambda: query_b_polars_eager(pldf)),
                ("C", lambda: query_c_polars_eager(pldf))]:
    print(f"\nRunning Polars eager query {qid}...")
    out_holder = {}
    def _run():
        out_holder["out"] = qfn()
        return out_holder["out"]
    m = measure(_run, repeats=2, warmup=1)
    add_result("polars_eager", qid, None, m, out_holder["out"].height)

# Polars lazy
for qid, qfn in [("A", lambda: query_a_polars_lazy(plscan)),
                ("B", lambda: query_b_polars_lazy(plscan)),
                ("C", lambda: query_c_polars_lazy(plscan))]:
    print(f"\nRunning Polars lazy query {qid}...")
    out_holder = {}
    def _run():
        out_holder["out"] = qfn()
        return out_holder["out"]
    m = measure(_run, repeats=2, warmup=1)
    add_result("polars_lazy", qid, None, m, out_holder["out"].height)

# DuckDB (threads sweep)
con = duckdb.connect()
for threads in [1, 2, 4, 8]:
    try:
        con.execute(f"PRAGMA threads={threads};")
    except Exception as e:
        print("DuckDB PRAGMA threads failed:", e)
        threads = None

    for qid, sql in [("A", DUCK_A), ("B", DUCK_B), ("C", DUCK_C)]:
        print(f"\nRunning DuckDB query {qid} (threads={threads})...")
        out_holder = {}
        def _run():
            out_holder["out"] = con.execute(sql).fetchdf()
            return out_holder["out"]
        m = measure(_run, repeats=2, warmup=1)
        add_result("duckdb", qid, threads, m, len(out_holder["out"]))

# Spark local scaling
spark_results = []
for cores in [1, 2, 4, 8]:
    print(f"\nStarting Spark local[{cores}] ...")
    spark = build_spark(cores)
    sdf = spark.read.parquet(DATA_PATH)

    for qid, qfn in [("A", lambda: spark_query_a(sdf)),
                    ("B", lambda: spark_query_b(sdf)),
                    ("C", lambda: spark_query_c(sdf))]:
        print(f"Running Spark query {qid} (local[{cores}]) ...")
        out_holder = {}
        def _run():
            out_df = qfn()
            # Force execution
            rows = out_df.count()
            out_holder["rows"] = rows
            return rows

        m = measure(_run, repeats=2, warmup=1)
        add_result("spark_local", qid, cores, m, out_holder["rows"])

    spark.stop()

# ---------------------------
# Show results table + save
# ---------------------------
res_df = pd.DataFrame(results)
display(res_df.sort_values(["query","engine","cores"], na_position="last"))

res_df.to_csv("phase2_task1_results.csv", index=False)
print("\nSaved: phase2_task1_results.csv")


Loading Pandas + Polars eager frames...
Rows (pandas): 5000000 Rows (polars): 5000000

Running Pandas query A...

Running Pandas query B...

Running Pandas query C...

Running Polars eager query A...


/tmp/ipykernel_7639/491526634.py:108: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("posts"),



Running Polars eager query B...

Running Polars eager query C...


/tmp/ipykernel_7639/491526634.py:128: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("posts"),



Running Polars lazy query A...


...


Starting Spark local[4] ...
Running Spark query A (local[4]) ...
Running Spark query B (local[4]) ...


Running Spark query C (local[4]) ...

Starting Spark local[8] ...
Running Spark query A (local[8]) ...
Running Spark query B (local[8]) ...


Running Spark query C (local[8]) ...


,engine,query,cores,time_min_s,time_mean_s,peak_mem_mb,rows_out
9,duckdb,A,1.0,0.777024,0.920826,10262.753906,49
12,duckdb,A,2.0,0.662753,0.796364,10115.605469,49
15,duckdb,A,4.0,0.654645,0.699400,10020.890625,49
18,duckdb,A,8.0,0.578560,0.669620,10023.957031,49
0,pandas,A,NaN,3.922133,8.056984,7264.535156,49
3,polars_eager,A,NaN,1.561501,2.047509,7773.699219,49
6,polars_lazy,A,NaN,0.641663,0.670186,9923.253906,49
21,spark_local,A,1.0,2.029102,3.018777,9892.363281,49
24,spark_local,A,2.0,0.946824,1.088145,9892.488281,49
27,spark_local,A,4.0,0.891644,0.932039,9892.488281,49



Saved: phase2_task1_results.csv


## Wnioski
Przeprowadzone eksperymenty na zbiorze danych o wielkości ok. **5 mln rekordów** zapisanych w formacie Parquet wyraźnie pokazują, że różnice w modelach obliczeniowych poszczególnych silników mają bezpośredni wpływ na czas wykonania zapytań oraz zużycie pamięci operacyjnej.

Pandas okazał się najmniej wydajnym rozwiązaniem we wszystkich testowanych scenariuszach. Czasy wykonania zapytań A, B i C mieściły się w przedziale **3–6 s**, przy jednoczesnym zużyciu pamięci rzędu **~7 GB RAM**. Wynika to z faktu, że Pandas zawsze ładuje cały zbiór danych do pamięci i nie wykorzystuje wielowątkowości, co czyni go nieodpowiednim narzędziem do pracy z dużymi wolumenami danych.

Polars zapewnił znaczną poprawę wydajności względem Pandas. W trybie eager czasy wykonania zapytań spadły do około **1–3 s**, natomiast w trybie lazy dla zapytań agregacyjnych (Query A) czas skrócił się nawet do **~0,6–0,7 s**. Odbywa się to jednak kosztem większego zużycia pamięci – w testach Polars zużywał od **~8 GB do ponad 10 GB RAM**. Wyniki te potwierdzają, że Polars efektywnie wykorzystuje równoległość CPU oraz optymalizację zapytań, lecz nadal jest rozwiązaniem typowo in-memory.

DuckDB wyróżniał się bardzo dobrą wydajnością w zapytaniach agregacyjnych i typu join. Dla Query A i C czasy wykonania wynosiły około **0,5–1,0 s**, a przy zwiększeniu liczby rdzeni z 1 do 8 obserwowano dalsze skrócenie czasu (np. Query A: **~0,78 s → ~0,58 s**). Jest to efekt pracy bezpośrednio na plikach Parquet i wykorzystania optymalizatora SQL, który minimalizuje ilość faktycznie przetwarzanych danych. Jednocześnie zapytania z funkcjami okienkowymi (Query B) pozostawały kosztowne – nawet przy 8 rdzeniach czas wykonania wynosił około **~2,1–2,3 s**, co pokazuje, że operacje wymagające sortowania danych są wąskim gardłem także dla DuckDB.

Spark w trybie single-node charakteryzował się wyraźnym narzutem startowym. Dla jednego rdzenia czasy zapytań mieściły się w przedziale **~1,5–3,5 s**, a dopiero przy zwiększeniu liczby rdzeni do 8 widoczne było istotne przyspieszenie (np. Query A: **~2,0 s → ~0,6 s**). W przypadku zapytań okienkowych (Query B) skalowanie było nieregularne – dla 2 i 4 rdzeni czasy były nawet gorsze niż dla jednego rdzenia, co wskazuje na dominację kosztów planowania i synchronizacji nad zyskiem z równoległości. Zużycie pamięci przez Spark utrzymywało się na poziomie **~9–10 GB RAM**, co jest typowe dla środowiska opartego o JVM.

Podsumowując, wyniki ilościowe potwierdzają, że dla przetwarzania danych mieszczących się na jednym węźle najbardziej efektywnymi rozwiązaniami są **Polars (czas wykonania rzędu setek milisekund do 1–2 s)** oraz **DuckDB (szczególnie dla zapytań SQL/ETL, ~0,5–1 s)**. Pandas sprawdza się jedynie dla małych zbiorów danych, natomiast Spark, mimo relatywnie słabych wyników w trybie lokalnym, pozostaje uzasadnionym wyborem w scenariuszach wymagających przetwarzania rozproszonego na klastrze.

### Task 2: Spark on Cluster

**Goal**: Compare Single Node performance vs. Spark on a Cluster.

**Instructions**:
1.  **Infrastructure**: Use the infrastructure from **Phase 1** (Google Dataproc). You may need to modify your Terraform code to adjust the cluster configuration (e.g., number of worker nodes).
2.  **Environment**: The easiest way to run this is via **Google Workbench** connected to your Dataproc cluster.
3.  **Upload Data**: Upload the generated `social_media_data.parquet` to HDFS or GCS.
    -   **Tip**: For better performance, consider **partitioning** the data (e.g., by `category` or `date`) when saving it to the distributed storage. This allows Spark to optimize reads.
4.  **Run Queries**: Run your PySpark queries from Task 1 on the cluster.
5.  **Scalability Test**: 
    -   Run the queries with different numbers of **worker nodes** (e.g., 2, 3, 4).
    -   You can achieve this by resizing the cluster (manually or via Terraform) or by configuring the number of executors in Spark.
6.  **Analyze**:
    -   How does the cluster performance compare to your local machine?
    -   Did adding more nodes/executors linearly improve performance?
    -   **Tip**: If Spark is slower than single-node engines, consider **increasing the dataset size** (e.g., generate 10M+ records or duplicate the data). Spark's overhead is significant for small data, and its true power appears when data exceeds single-node memory.

In [6]:
# Your Code Here for Task 2

### Task 3: Execution Modes & Analysis

**Goal**: Deep dive into execution models and limitations.

**Instructions**:
1.  **Lazy vs. Eager vs. Streaming**:
    -   Use **Polars**. Compare the **Execution Time** and **Peak Memory** of:
        -   Eager execution (`read_parquet` -> filter).
        -   Lazy execution (`scan_parquet` -> filter -> `collect()`).
        -   Streaming execution (`scan_parquet` -> filter -> `collect(streaming=True)`).
2.  **Polars Limitations**:
    -   Identify a scenario where Polars might struggle compared to Spark (e.g., memory limits).
3.  **Decision Boundary**:
    -   Based on your findings, when would you recommend switching from a single-node tool (Polars/DuckDB) to a distributed engine (Spark)?

In [7]:
# Your Code Here for Task 3